In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# 1. Fetch and Flatten the Data
ticker = "RELIANCE.NS" 
df = yf.download(ticker, start="2020-01-01", end="2024-01-01")
data = pd.DataFrame()
data['Close'] = df['Close'].squeeze()
data.dropna(inplace=True)

# 2. Mathematical Logic
data['SMA20'] = data['Close'].rolling(window=20).mean()
data['SMA50'] = data['Close'].rolling(window=50).mean()
data['Signal'] = np.where(data['SMA20'] > data['SMA50'], 1.0, 0.0)
data['Position'] = data['Signal'].diff()

# 3. Interactive Jupyter Visualization
fig = go.Figure()

fig.add_trace(go.Scatter(x=data.index, y=data['Close'], mode='lines', name='Close', line=dict(color='rgba(0,0,0,0.3)')))
fig.add_trace(go.Scatter(x=data.index, y=data['SMA20'], mode='lines', name='SMA20', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=data.index, y=data['SMA50'], mode='lines', name='SMA50', line=dict(color='orange')))

# Buy/Sell Markers
buys = data[data['Position'] == 1.0]
sells = data[data['Position'] == -1.0]
fig.add_trace(go.Scatter(x=buys.index, y=buys['Close'], mode='markers', name='BUY', marker=dict(color='green', size=10, symbol='triangle-up')))
fig.add_trace(go.Scatter(x=sells.index, y=sells['Close'], mode='markers', name='SELL', marker=dict(color='red', size=10, symbol='triangle-down')))

fig.update_layout(
    title=f"{ticker} Crossover Strategy",
    hovermode="x unified",
    xaxis=dict(rangeslider=dict(visible=True)),
    height=600 # Fits nicely inside a notebook cell
)

# In Jupyter, this will embed the interactive chart directly below the cell!
fig.show()

[*********************100%***********************]  1 of 1 completed


In [2]:
# 1. Calculate daily returns of the asset
data['Asset_Return'] = data['Close'].pct_change()

# 2. Calculate strategy returns (shifting the signal to avoid look-ahead bias)
data['Strategy_Return'] = data['Asset_Return'] * data['Signal'].shift(1)

# Calculate total cumulative growth (assuming compounding)
cum_asset = (1 + data['Asset_Return']).cumprod().iloc[-1] - 1
cum_strategy = (1 + data['Strategy_Return']).cumprod().iloc[-1] - 1

# Print the final results
print(f"Buy & Hold Return: {cum_asset * 100:.2f}%")
print(f"Strategy Return:   {cum_strategy * 100:.2f}%")

NameError: name 'data' is not defined